# 02 — Comparator Source Triage and RA Review Workbook

Purpose: convert the completed comparator deep-search run into a researcher/RA-facing workbook. This notebook does not treat deep-search synthesis claims as findings. It ranks likely useful comparator sources, flags citation risks, and creates a metric-coding template for human verification.

Run this after `Deep_Search_Construction_Asset_Comparators_FRESH.ipynb` has completed and produced `FINAL_MERGED_SYNTHESIS.md`, `verified_sources.csv`, `datasets.csv`, and `QA_REPORT.md`.

In [ ]:
# Optional first-time setup:
# %pip install -q pandas openpyxl

from pathlib import Path
from datetime import datetime
import json
import math
import re
import textwrap
import pandas as pd

# Leave blank to use the most recent completed comparator run.
RUN_TAG = ""

# Controls how many sources are carried into the RA metric-review sheet.
MAX_RA_SOURCES = 90
MIN_SCORE_FOR_RA = 7


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    candidates.append(Path.cwd().resolve() / "DC_job_potential")
    for p in candidates:
        if (p / "Research_Workflow").is_dir() and (p.name == "DC_job_potential" or (p / "Media_Article").exists()):
            return p
    raise RuntimeError("Could not locate DC_job_potential root. Open the project folder or run this notebook from inside it.")


ROOT = find_project_root(Path.cwd())
WORKFLOW = ROOT / "Research_Workflow"
DEEP_SEARCH_ROOT = WORKFLOW / "01_Deep_Search" / "02_comparator_assets" / "deep_search_outputs"
EXTRACTION = WORKFLOW / "02_Source_Extraction"
DATA_DIR = EXTRACTION / "data"
REGISTER_DIR = DATA_DIR / "source_register"
OUT_DIR = DATA_DIR / "coding_outputs" / "comparator_review"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_PATH = REGISTER_DIR / "comparative_asset_coding_template.csv"

print("ROOT:", ROOT)
print("DEEP_SEARCH_ROOT:", DEEP_SEARCH_ROOT)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def latest_completed_run(base: Path, run_tag: str = "") -> Path:
    if run_tag:
        run = base / run_tag
        if not run.exists():
            raise FileNotFoundError(f"Requested run tag not found: {run}")
        return run
    runs = sorted(
        [p for p in base.glob("run_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for run in runs:
        if (run / "FINAL_MERGED_SYNTHESIS.md").exists() and (run / "QA_REPORT.md").exists():
            return run
    raise FileNotFoundError("No completed comparator deep-search run found.")


RUN_DIR = latest_completed_run(DEEP_SEARCH_ROOT, RUN_TAG)
RUN_TAG = RUN_DIR.name

paths = {
    "final_synthesis": RUN_DIR / "FINAL_MERGED_SYNTHESIS.md",
    "qa_report": RUN_DIR / "QA_REPORT.md",
    "verified_sources": RUN_DIR / "verified_sources.csv",
    "datasets": RUN_DIR / "datasets.csv",
    "citation_audit": RUN_DIR / "final_citation_audit.csv",
    "checkpoint": RUN_DIR / "checkpoint.json",
}

for label, path in paths.items():
    print(f"{label:18s}", path.exists(), path.name)

synthesis_text = paths["final_synthesis"].read_text(encoding="utf-8", errors="ignore")
qa_text = paths["qa_report"].read_text(encoding="utf-8", errors="ignore")
verified_sources = pd.read_csv(paths["verified_sources"]) if paths["verified_sources"].exists() else pd.DataFrame()
datasets = pd.read_csv(paths["datasets"]) if paths["datasets"].exists() else pd.DataFrame()
citation_audit = pd.read_csv(paths["citation_audit"]) if paths["citation_audit"].exists() else pd.DataFrame()

print("\nLoaded:")
print("verified_sources:", len(verified_sources))
print("datasets:", len(datasets))
print("citation_audit:", len(citation_audit))

In [ ]:
def clean_cell(value):
    value = str(value).strip()
    value = re.sub(r"\*\*|<br\s*/?>|`", "", value)
    return re.sub(r"\s+", " ", value).strip()


def extract_markdown_table_after_heading(text: str, heading: str) -> pd.DataFrame:
    lines = text.splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith(heading.lower()):
            start = i + 1
            break
    if start is None:
        return pd.DataFrame()

    table_lines = []
    in_table = False
    for line in lines[start:]:
        if line.startswith("## ") and in_table:
            break
        if line.strip().startswith("|"):
            in_table = True
            table_lines.append(line.strip())
        elif in_table and line.strip() == "":
            continue
        elif in_table:
            break

    if len(table_lines) < 3:
        return pd.DataFrame()

    header = [clean_cell(x) for x in table_lines[0].strip("|").split("|")]
    rows = []
    for line in table_lines[2:]:
        parts = [clean_cell(x) for x in line.strip("|").split("|")]
        if len(parts) == len(header):
            rows.append(dict(zip(header, parts)))
    return pd.DataFrame(rows)


def extract_doi(value: str) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    m = re.search(r"10\.\d{4,9}/[^\s,;|)]+", str(value), flags=re.I)
    return m.group(0).rstrip(".]").lower() if m else ""


def extract_url(value: str) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    m = re.search(r"https?://[^\s)]+", str(value), flags=re.I)
    return m.group(0).rstrip(".]") if m else ""


synthesis_evidence = extract_markdown_table_after_heading(synthesis_text, "## 6) Master Evidence Table")
print("synthesis evidence rows:", len(synthesis_evidence))
synthesis_evidence.head(20)

In [ ]:
ASSET_KEYWORDS = {
    "high_rise_or_office": ["high-rise", "high rise", "tower", "office", "commercial building", "mixed-use", "residential"],
    "hospital_or_social_infrastructure": ["hospital", "healthcare", "health care", "clinic", "social infrastructure"],
    "warehouse_or_logistics": ["warehouse", "logistics", "fulfilment", "fulfillment", "distribution centre", "distribution center", "freight"],
    "advanced_manufacturing_or_fab": ["semiconductor", "fab", "chip", "battery", "gigafactory", "advanced manufacturing", "manufacturing employment"],
    "energy_or_utility": ["renewable", "solar", "wind", "power plant", "transmission", "grid", "energy efficiency", "megawatt", "mw"],
    "public_infrastructure_or_methods": ["input-output", "input output", "cge", "implan", "rims", "multiplier", "public infrastructure", "construction sector", "regional", "economic impact"],
    "data_centre_benchmark": ["data center", "data centre", "hyperscale", "cloud", "digital infrastructure"],
}

METRIC_KEYWORDS = [
    "job", "jobs", "employment", "worker", "workers", "fte", "job-year", "job year", "labour", "labor",
    "capex", "capital expenditure", "investment", "multiplier", "economic impact", "operational", "construction",
    "gross floor", "floor area", "gfa", "mw", "megawatt", "permanent", "direct", "indirect", "induced",
]

STRONG_SOURCE_HINTS = ["government", "audit", "treasury", "infrastructure australia", "bea", "abs", "jlarc", "nber", "journal", "working paper"]
BIAS_HINTS = ["coalition", "industry", "sponsored", "cbre", "jll", "pwc", "mandala", "uptime institute", "consultant"]
WEAK_OR_OFF_TOPIC_HINTS = ["children", "language", "tourism", "food insecurity", "unemployment benefits", "author guidelines", "weather on economic growth"]


def normalize_text(*values) -> str:
    return " ".join("" if pd.isna(v) else str(v) for v in values).lower()


def guess_asset_type(blob: str) -> str:
    hits = []
    for asset, terms in ASSET_KEYWORDS.items():
        if any(term in blob for term in terms):
            hits.append(asset)
    return "; ".join(hits) if hits else "unclear_or_methods_only"


def evidence_role(blob: str) -> str:
    roles = []
    if any(t in blob for t in ["job-year", "job year", "construction jobs", "construction employment", "worker"]):
        roles.append("construction_jobs")
    if any(t in blob for t in ["operational jobs", "permanent jobs", "fte", "employment density", "staff"]):
        roles.append("operational_jobs")
    if any(t in blob for t in ["capex", "capital expenditure", "investment", "$", "a$", "us$"]):
        roles.append("capex")
    if any(t in blob for t in ["mw", "megawatt", "capacity"]):
        roles.append("capacity_mw")
    if any(t in blob for t in ["multiplier", "input-output", "input output", "implan", "cge", "rims"]):
        roles.append("method_or_multiplier")
    if any(t in blob for t in ["audit", "realised", "realized", "post-completion", "ex-post", "ex post"]):
        roles.append("realised_or_audit")
    return "; ".join(roles) if roles else "context_only"


def source_type_guess(row) -> str:
    blob = normalize_text(row.get("title"), row.get("container_or_repository"), row.get("url"), row.get("source_origin"))
    if "gov" in blob or any(t in blob for t in ["treasury", "infrastructure australia", "jlarc", "abs", "bea"]):
        return "government_or_audit"
    if row.get("doi", ""):
        return "doi_verified_academic_or_report"
    if "dataset" in blob or "zenodo" in blob:
        return "dataset_or_repository_record"
    if any(t in blob for t in BIAS_HINTS):
        return "industry_or_consultant_report"
    return "web_or_other"


def bias_flag(row) -> str:
    blob = normalize_text(row.get("title"), row.get("container_or_repository"), row.get("url"), row.get("source_origin"))
    flags = []
    if any(t in blob for t in BIAS_HINTS):
        flags.append("possible_industry_or_consultant_bias")
    if "citation_audit_invalid" in blob:
        flags.append("citation_audit_invalid_or_mismatch")
    if any(t in blob for t in WEAK_OR_OFF_TOPIC_HINTS):
        flags.append("likely_off_topic")
    return "; ".join(flags) if flags else ""


def score_source(row) -> tuple[int, str]:
    blob = normalize_text(row.get("title"), row.get("container_or_repository"), row.get("evidence_note"), row.get("url"), row.get("source_origin"))
    score = 0
    reasons = []
    if "final_synthesis_master_table" in blob:
        score += 8; reasons.append("appears in final synthesis evidence table")
    if row.get("citation_audit_status") == "verified":
        score += 4; reasons.append("citation audit verified")
    if row.get("citation_audit_status") == "invalid":
        score -= 8; reasons.append("citation audit invalid")
    if row.get("doi", ""):
        score += 3; reasons.append("has DOI")
    asset_hit = guess_asset_type(blob)
    if asset_hit != "unclear_or_methods_only":
        score += 4; reasons.append("asset-class keywords found")
    if any(t in blob for t in METRIC_KEYWORDS):
        score += 4; reasons.append("employment/capex/normalisation keywords found")
    if any(t in blob for t in STRONG_SOURCE_HINTS):
        score += 3; reasons.append("authoritative or scholarly source hint")
    if any(t in blob for t in WEAK_OR_OFF_TOPIC_HINTS):
        score -= 5; reasons.append("weak/off-topic title hint")
    return score, "; ".join(reasons)


print("Scoring helpers ready.")

In [ ]:
def slug(value: str, limit: int = 50) -> str:
    out = re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")
    return out[:limit] or "source"


def standardize_verified(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    out = pd.DataFrame({
        "title": df.get("title", ""),
        "year": df.get("year", ""),
        "container_or_repository": df.get("container", ""),
        "authors_or_creators": df.get("authors", ""),
        "doi": df.get("doi", "").astype(str).str.lower(),
        "url": df.get("URL", ""),
        "source_origin": "verified_sources",
        "evidence_note": "",
    })
    return out


def standardize_datasets(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    out = pd.DataFrame({
        "title": df.get("title", ""),
        "year": df.get("year", ""),
        "container_or_repository": df.get("repository", ""),
        "authors_or_creators": df.get("creators", ""),
        "doi": df.get("doi", "").fillna("").astype(str).str.lower(),
        "url": df.get("url", ""),
        "source_origin": "datasets",
        "evidence_note": df.get("query", "").fillna("").astype(str) + " | " + df.get("suitability", "").fillna("").astype(str),
    })
    return out


def standardize_synthesis_table(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    title_col = next((c for c in df.columns if "source" in c.lower() or "author" in c.lower()), df.columns[0])
    asset_col = next((c for c in df.columns if "asset" in c.lower()), None)
    finding_col = next((c for c in df.columns if "finding" in c.lower() or "benchmark" in c.lower()), None)
    quality_col = next((c for c in df.columns if "quality" in c.lower() or "bias" in c.lower()), None)
    url_col = next((c for c in df.columns if "doi" in c.lower() or "url" in c.lower()), None)
    raw_ref = df[url_col] if url_col else ""
    out = pd.DataFrame({
        "title": df[title_col].astype(str),
        "year": "",
        "container_or_repository": "final synthesis evidence table",
        "authors_or_creators": "",
        "doi": df[url_col].map(extract_doi).str.lower() if url_col else "",
        "url": df[url_col].map(lambda x: extract_url(x) or (f"https://doi.org/{extract_doi(x)}" if extract_doi(x) else str(x))) if url_col else "",
        "source_origin": "final_synthesis_master_table",
        "evidence_note": "",
    })
    notes = []
    for _, r in df.iterrows():
        parts = []
        if asset_col: parts.append(f"Asset: {r.get(asset_col, '')}")
        if finding_col: parts.append(f"Claim/benchmark to verify: {r.get(finding_col, '')}")
        if quality_col: parts.append(f"Quality note: {r.get(quality_col, '')}")
        notes.append(" | ".join(parts))
    out["evidence_note"] = notes
    return out


pool = pd.concat(
    [standardize_verified(verified_sources), standardize_datasets(datasets), standardize_synthesis_table(synthesis_evidence)],
    ignore_index=True,
)
pool["doi"] = pool["doi"].fillna("").astype(str).str.lower().str.strip()
pool["url"] = pool["url"].fillna("").astype(str).str.strip()
pool["title"] = pool["title"].fillna("").astype(str).str.strip()

audit_map = {}
if not citation_audit.empty and "doi" in citation_audit.columns and "status" in citation_audit.columns:
    audit_map = dict(zip(citation_audit["doi"].fillna("").astype(str).str.lower(), citation_audit["status"].fillna("")))
pool["citation_audit_status"] = pool["doi"].map(audit_map).fillna("")

pool["dedupe_key"] = pool.apply(lambda r: r["doi"] or r["url"].lower() or slug(r["title"], 120), axis=1)

def first_nonempty(series):
    for value in series:
        if not pd.isna(value) and str(value).strip():
            return value
    return ""


triage = pool.groupby("dedupe_key", as_index=False).agg({
    "title": first_nonempty,
    "year": first_nonempty,
    "container_or_repository": first_nonempty,
    "authors_or_creators": first_nonempty,
    "doi": first_nonempty,
    "url": first_nonempty,
    "source_origin": lambda s: "; ".join(sorted(set(str(x) for x in s if str(x).strip()))),
    "evidence_note": lambda s: " | ".join([str(x) for x in s if str(x).strip()])[:1200],
    "citation_audit_status": first_nonempty,
})

scores = triage.apply(score_source, axis=1)
triage["relevance_score"] = [s[0] for s in scores]
triage["score_reason"] = [s[1] for s in scores]
triage["asset_type_guess"] = triage.apply(lambda r: guess_asset_type(normalize_text(r["title"], r["container_or_repository"], r["evidence_note"], r["url"])), axis=1)
triage["evidence_role_guess"] = triage.apply(lambda r: evidence_role(normalize_text(r["title"], r["container_or_repository"], r["evidence_note"], r["url"])), axis=1)
triage["source_type_guess"] = triage.apply(source_type_guess, axis=1)
triage["funding_or_bias_flag"] = triage.apply(bias_flag, axis=1)
triage["priority"] = pd.cut(triage["relevance_score"], bins=[-999, 6, 10, 999], labels=["low", "medium", "high"])
triage["include_for_RA"] = triage["relevance_score"].ge(MIN_SCORE_FOR_RA) | triage["source_origin"].str.contains("final_synthesis_master_table", na=False)
triage["ra_next_action"] = triage.apply(
    lambda r: "verify cited benchmark and extract metric/page" if r["include_for_RA"] else "keep as background unless manually promoted",
    axis=1,
)
triage = triage.sort_values(["include_for_RA", "relevance_score", "title"], ascending=[False, False, True]).reset_index(drop=True)
triage.insert(0, "triage_id", [f"cmp_src_{i+1:04d}" for i in range(len(triage))])

print("Total unique source records:", len(triage))
print("Included for RA:", int(triage["include_for_RA"].sum()))
triage.head(25)

In [ ]:
template_cols = list(pd.read_csv(TEMPLATE_PATH, nrows=0).columns) if TEMPLATE_PATH.exists() else [
    "record_id", "source_id", "asset_type", "project_or_source_name", "location", "country", "year", "source_type", "method",
    "funding_or_bias_flag", "capex_value", "capex_currency", "capex_price_year", "capex_aud_2026", "gfa_m2", "capacity_mw",
    "construction_jobs_peak", "construction_jobs_average", "construction_duration_months", "construction_job_years", "operations_jobs_fte",
    "direct_jobs_only", "total_supported_jobs", "projected_or_realised", "reported_or_derived", "metric_notes", "source_quote",
    "page_or_location", "source_url", "full_text_file", "confidence", "ra_verification_status",
]

ra_sources = triage[triage["include_for_RA"]].sort_values("relevance_score", ascending=False).head(MAX_RA_SOURCES).copy()
review_rows = []
for _, r in ra_sources.iterrows():
    source_id = r["triage_id"]
    review_rows.append({
        "record_id": f"{source_id}_metric_001",
        "source_id": source_id,
        "asset_type": r["asset_type_guess"],
        "project_or_source_name": r["title"],
        "location": "",
        "country": "",
        "year": r["year"],
        "source_type": r["source_type_guess"],
        "method": "",
        "funding_or_bias_flag": r["funding_or_bias_flag"],
        "metric_notes": "RA task: open source, verify whether it reports construction jobs, job-years, operational FTE, capex, GFA, MW/capacity, multiplier method, and direct-vs-total job definition. " + str(r["evidence_note"]),
        "source_quote": "",
        "page_or_location": "",
        "source_url": r["url"],
        "full_text_file": "",
        "confidence": "unverified_candidate_source",
        "ra_verification_status": "not_started",
    })

metric_review = pd.DataFrame(review_rows)
for col in template_cols:
    if col not in metric_review.columns:
        metric_review[col] = ""
metric_review = metric_review[template_cols]

citation_flags = citation_audit.copy() if not citation_audit.empty else pd.DataFrame(columns=["doi", "status", "title", "year", "url"])
if not citation_flags.empty:
    citation_flags["action"] = citation_flags["status"].map({
        "verified": "usable after source-level claim verification",
        "invalid": "do not cite until manually repaired or replaced",
    }).fillna("check manually")

readme = pd.DataFrame([
    {"field": "run_tag", "value": RUN_TAG},
    {"field": "created_at", "value": datetime.now().isoformat(timespec="seconds")},
    {"field": "purpose", "value": "Triage comparator deep-search sources and create RA metric-review template."},
    {"field": "rule", "value": "A row is not a finding until the RA/researcher verifies the source page/table, metric definition, denominator, and direct-vs-total job concept."},
    {"field": "recommended_next_step", "value": "Start with high-priority rows in metric_review_template, especially final-synthesis evidence-table rows and government/audit/method sources."},
])

csv_triage = OUT_DIR / "comparator_source_triage.csv"
csv_review = OUT_DIR / "comparator_metric_review_template.csv"
xlsx_path = OUT_DIR / "comparator_RA_review_workbook.xlsx"

triage.to_csv(csv_triage, index=False)
metric_review.to_csv(csv_review, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    readme.to_excel(writer, sheet_name="README", index=False)
    triage.to_excel(writer, sheet_name="source_triage", index=False)
    metric_review.to_excel(writer, sheet_name="metric_review_template", index=False)
    citation_flags.to_excel(writer, sheet_name="citation_quality_flags", index=False)
    synthesis_evidence.to_excel(writer, sheet_name="synthesis_evidence_table", index=False)
    verified_sources.to_excel(writer, sheet_name="verified_sources_raw", index=False)
    datasets.to_excel(writer, sheet_name="datasets_raw", index=False)

# Light formatting for RA usability.
try:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation

    wb = load_workbook(xlsx_path)
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.fill = PatternFill("solid", fgColor="D9EAF7")
        for col in ws.columns:
            header = str(col[0].value or "")
            width = min(max(len(header) + 4, 14), 45)
            ws.column_dimensions[col[0].column_letter].width = width

    if "metric_review_template" in wb.sheetnames:
        ws = wb["metric_review_template"]
        headers = {cell.value: cell.column_letter for cell in ws[1]}
        if "ra_verification_status" in headers:
            dv = DataValidation(type="list", formula1='"not_started,in_progress,verified,rejected,needs_source"', allow_blank=True)
            ws.add_data_validation(dv)
            dv.add(f"{headers['ra_verification_status']}2:{headers['ra_verification_status']}{ws.max_row}")
        if "confidence" in headers:
            dv = DataValidation(type="list", formula1='"unverified_candidate_source,verified_high,verified_medium,verified_low,rejected"', allow_blank=True)
            ws.add_data_validation(dv)
            dv.add(f"{headers['confidence']}2:{headers['confidence']}{ws.max_row}")
    wb.save(xlsx_path)
except Exception as e:
    print("Workbook formatting skipped:", e)

print("Wrote:")
print("-", csv_triage)
print("-", csv_review)
print("-", xlsx_path)
print("\nWorkbook rows:")
print("source_triage:", len(triage))
print("metric_review_template:", len(metric_review))
print("citation_quality_flags:", len(citation_flags))

## Interpretation Notes

- `source_triage` is broad: it keeps traceability to the deep-search run and explains why each source was promoted or left in the background.
- `metric_review_template` is the RA-facing coding sheet. The RA should only fill a metric after opening the source and recording the exact page/table/quote.
- `citation_quality_flags` should be checked before any source is cited. Rows marked invalid should be repaired or replaced.
- Deep-search synthesis claims are treated as leads, not as article-ready evidence.